# 5 — Auditing a compiled model for fair lending

Three layers, eleven sections, over decisions the artifact already made.

**What this produces is evidence, not a verdict.** Nothing here certifies
compliance with ECOA, Regulation B, or anything else. It gives a reviewer
numbers they can inspect and argue with, which is the most a tool should
claim.

The first two layers — approval ratios, score distributions, error rates,
calibration — are commodity. `fairlearn` and `aif360` do them well. The third
layer is why this lives in CompileML: two of its sections are only answerable
because a compiled artifact carries **exact attributions** and the **real
reason codes**, rather than approximations of either.

## Data

The UCI credit-default panel has a genuine protected attribute, which makes it the honest example. It is fetched once and cached; if it is unavailable — CI runs offline — we fall back to synthetic data so the notebook still runs, and say which was used.

In [1]:
import warnings

import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

from compileml.artifact import build_artifact
from compileml.bands import quantile_bands
from compileml.compile import train_whitebox
from compileml.datasets import load_credit_default, make_credit_data
from compileml.fairness import FairnessAudit, model_interaction_structure
from compileml.runtime import decide

SEED = 42

try:
    X, y, names = load_credit_default(include_demographics=True)
    protected = X[:, names.index("SEX")].astype(int)
    labels = {1: "Male", 2: "Female"}
    protected_feature = "SEX"          # it IS a model input here
    source = "UCI credit-default panel (real protected attribute)"
except Exception as exc:               # offline, e.g. CI
    X, y, names = make_credit_data(n_rows=12_000, n_features=12, seed=SEED)
    rng = np.random.default_rng(SEED)
    # A stand-in attribute correlated with a real feature, so the audit has
    # something to find. Not a substitute for real data — see the note below.
    protected = (X[:, 0] + rng.normal(0, 1.5, len(X)) > 0).astype(int) + 1
    labels = {1: "Group A", 2: "Group B"}
    protected_feature = None           # not a model input in this branch
    source = f"synthetic fallback ({type(exc).__name__})"

print(f"source      : {source}")
print(f"rows        : {len(X):,}   features: {len(names)}")
print(f"base rate   : {y.mean():.2%}")
print(f"group sizes : {dict(zip(*np.unique(protected, return_counts=True)))}")

source      : UCI credit-default panel (real protected attribute)
rows        : 30,000   features: 23
base rate   : 22.12%
group sizes : {np.int64(1): np.int64(11888), np.int64(2): np.int64(18112)}


## Compile

Stratify on outcome **and** group jointly, so neither split drifts. Then the ordinary path: teacher, distilled whitebox at depth 2, artifact.

In [2]:
strat = y * 10 + (protected - protected.min())
X_tr, X_te, y_tr, y_te, g_tr, g_te = train_test_split(
    X, y, protected, test_size=0.25, random_state=SEED, stratify=strat
)

teacher = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, random_state=SEED
).fit(X_tr, y_tr)

whitebox, metrics = train_whitebox(
    X_tr, teacher.predict_proba(X_tr)[:, 1],
    n_estimators=40, max_depth=2, random_state=SEED,
)
latent = np.clip(whitebox.predict(X_tr), 0.0, 1.0)

reasons = {
    n: {"code": n.upper(), "negative": f"{n} raised risk", "positive": f"{n} lowered risk"}
    for n in names
}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")     # reason coverage is complete here
    artifact = build_artifact(
        whitebox, names, np.median(X_tr, axis=0),
        quantile_bands(latent, n_bands=10),
        calibration_latent=latent, calibration_y=y_tr, reasons=reasons,
    )

print(f"depth {artifact['runtime']['whitebox_max_depth']} | "
      f"exact attribution: {artifact['runtime']['exact_attribution']}")

depth 2 | exact attribution: True


## Decide, then audit

`include_contributions=True` is **required** for sections 6 and 8. The default
payload carries `attribution` as a *method label* only — the per-feature
impacts live behind the flag, and the audit raises rather than guessing if
they are absent.

In [3]:
decisions = [
    decide(artifact, [float(v) for v in row], include_contributions=True)
    for row in X_te
]

audit = FairnessAudit(
    decisions, y_te, g_te,
    labels=labels,
    artifact=artifact,
    X=X_te,
    protected_feature=protected_feature,
)
audit.compute_all()
audit.print_summary()

Fairness audit

  Male  (n=2,972  39.6%)
    observed bad rate : 24.16%
    approval rate     : 46.87%  [45.08%, 48.67%]
    FPR / FNR         : 44.28% / 19.08%
    near the cutoff   : 19.2% within 25 points

  Female  (n=4,528  60.4%)
    observed bad rate : 20.78%
    approval rate     : 51.72%  [50.27%, 53.18%]
    FPR / FNR         : 40.37% / 21.57%
    near the cutoff   : 18.5% within 25 points

  adverse impact ratio (Male / Female): 0.906  — inside the 0.80-1.25 window

  mean score gap decomposed exactly (residual 0); largest drivers:
    PAY_1                     36.9% of the gap
    LIMIT_BAL                 15.9% of the gap
    SEX                       13.4% of the gap

  adverse-action reason divergence (total variation): 0.108

  This is evidence, not a verdict. It does not certify compliance
  with ECOA, Regulation B, or anything else.


`threshold_int` was not supplied, so the audit used the **median score** as a
cutoff. That is a reporting convenience, not a policy — every approval number
above moves with it. In a real audit, pass the cutoff you actually deploy.

---
## Layer 1 — what decisions were made

In [4]:
rep = audit.section(1)
for name, g in rep["groups"].items():
    lo, hi = g["base_rate_ci"]
    print(f"{name:<10} n={g['n']:>6,}  ({g['share']:.1%})   "
          f"bad rate {g['base_rate']:.2%}  [{lo:.2%}, {hi:.2%}]")

if "feature_divergence" in rep:
    print("\nInputs that differ most between groups (KS):")
    for row in rep["feature_divergence"][:5]:
        print(f"  {row['feature']:<12} {row['ks']:.4f}")

Male       n= 2,972  (39.6%)   bad rate 24.16%  [22.65%, 25.73%]
Female     n= 4,528  (60.4%)   bad rate 20.78%  [19.62%, 21.99%]

Inputs that differ most between groups (KS):
  SEX          1.0000
  LIMIT_BAL    0.0934
  AGE          0.0810
  PAY_AMT6     0.0565
  BILL_AMT2    0.0560


Those input divergences matter before anything else. A score gap fully
explained by an input gap is a different conversation from one that is not.

If the protected attribute appears at the top with a KS of 1.0, that is not a
bug and not noise — groups differ perfectly on the attribute that defines
them. Read it as a sanity check: it tells you the attribute is **in the
feature set**, which is exactly what §11 will test later.

In [5]:
appr = audit.section(3)
for name, a in appr["groups"].items():
    lo, hi = a["rate_ci"]
    print(f"{name:<10} approval {a['rate']:.2%}  [{lo:.2%}, {hi:.2%}]")
print(f"\nadverse impact ratio ({appr['ratio_of']}): {appr['adverse_impact_ratio']:.4f}")
print(f"inside the 0.80-1.25 window: {appr['within_four_fifths']}")

Male       approval 46.87%  [45.08%, 48.67%]
Female     approval 51.72%  [50.27%, 53.18%]

adverse impact ratio (Male / Female): 0.9062
inside the 0.80-1.25 window: True


---
## Layer 2 — how well it predicts per group

In [6]:
err = audit.section(5)
print(f"{'group':<10}{'FPR':>9}{'FNR':>9}{'precision':>12}{'recall':>9}")
for name, e in err["groups"].items():
    print(f"{name:<10}{e['fpr']:>8.2%}{e['fnr']:>9.2%}{e['precision']:>12.2%}{e['recall']:>9.2%}")

cal = audit.section(4)
print("\ncalibration bias (predicted minus observed):")
for name, c in cal["groups"].items():
    print(f"  {name:<10} {c['bias']:+.4f}   mean abs gap {c['mean_abs_calibration_gap']:.4f}")

group           FPR      FNR   precision   recall
Male        44.28%   19.08%      36.80%   80.92%
Female      40.37%   21.57%      33.76%   78.43%

calibration bias (predicted minus observed):
  Male       -0.0033   mean abs gap 0.0253
  Female     +0.0086   mean abs gap 0.0209


---
## Layer 3 — how it behaves locally

### §6 Attribution disparity — the headline

The mean group score gap, decomposed by feature. Watch the residual.

In [7]:
s6 = audit.section(6)
print(f"comparison : {s6['comparison']}")
print(f"mean gap   : {s6['mean_gap_half_micro']:>18,.6f}  (half-micro)")
print(f"sum of parts: {s6['sum_of_feature_gaps']:>17,.6f}")
print(f"residual   : {s6['residual']:>18.3e}")
print("\nlargest drivers of the gap:")
for row in s6["by_feature"][:6]:
    print(f"  {row['feature']:<12} {row['gap_half_micro']:>16,.0f}   {row['share_pct']:>7.2f}%")

comparison : Male minus Female
mean gap   :      39,396.625742  (half-micro)
sum of parts:     39,396.625742
residual   :          1.455e-11

largest drivers of the gap:
  PAY_1                  14,554     36.94%
  LIMIT_BAL               6,270     15.91%
  SEX                     5,290     13.43%
  PAY_2                   3,055      7.75%
  PAY_3                   2,758      7.00%
  PAY_6                   2,479      6.29%


### Why the residual is not literally `0.0`

It prints as something like `1.5e-11`, and that is worth being precise about:
the **per-row integer** contributions sum to the per-row integer movement
exactly, with no residual at all. What you see here is the float
representation of a *mean* over thousands of those exact integers. The
guarantee is on the integers; the `e-11` is IEEE-754 arithmetic on the
average, and it does not accumulate.

The residual is **zero**, not small. Because the artifact's attribution
reconciles to the score with no residual at depth ≤ 2, the per-feature
contributions to a group gap *sum to the gap*.

That is what turns *"the ratio is 0.91"* into *"37% of the gap is `PAY_1`"* —
an observation becomes a remediation plan.

A share above 100% is a real finding rather than an error: one driver widens
the gap further than observed while others partially offset it. No
decomposition whose parts fail to sum to the whole can state that.

Above depth 2 the residual is nonzero and this section **refuses** rather than
presenting an approximation as exact.

### §9 Boundary fragility

An adverse impact ratio is a snapshot. This is its derivative: who is close enough to the cutoff that a small policy change moves them.

In [8]:
frag = audit.section(9)
for name, f in frag["groups"].items():
    print(f"{name:<10} mean distance {f['mean_distance']:>7.1f}   "
          f"{f['share_within_band']:.1%} within {frag['near_band']} points of the cutoff")

Male       mean distance   128.2   19.2% within 25 points of the cutoff
Female     mean distance   116.9   18.5% within 25 points of the cutoff


### §10 Adverse-action reason parity

This is regulatory rather than statistical — it is about what applicants are
**told**.

Note what is being counted: `reasons_negative` *is* the adverse-action code
the applicant would be sent, produced by the same runtime that made the
decision. Not a feature-importance ranking standing in for it.

In [9]:
rp = audit.section(10)
for name, g in rp["groups"].items():
    top = ", ".join(f"{r['feature']} {r['share']:.0%}" for r in g["top"][:4])
    print(f"{name:<10} {top}")
print(f"\ntotal variation between the two reason distributions: {rp['total_variation']:.4f}")
if rp.get("divergence"):
    print("\nlargest differences in cited reason:")
    for d in rp["divergence"][:4]:
        print(f"  {d['feature']:<12} delta {d['delta']:+.4f}")

Male       LIMIT_BAL 19%, PAY_1 17%, PAY_2 9%, MARRIAGE 8%
Female     PAY_1 18%, LIMIT_BAL 15%, MARRIAGE 10%, PAY_AMT1 10%

total variation between the two reason distributions: 0.1078

largest differences in cited reason:
  SEX          delta +0.0427
  PAY_AMT1     delta -0.0383
  LIMIT_BAL    delta +0.0381
  PAY_AMT2     delta -0.0179


### §8 Attribution concentration

How many drivers it takes to explain a decision. Reported as the top driver's
share and an effective count, `exp(entropy)`, which reads directly.

In [10]:
s8 = audit.section(8)
for name, g in s8["groups"].items():
    print(f"{name:<10} top driver {g['top_driver_share']:.1%}   "
          f"effective drivers {g['effective_drivers']:.2f}")
if "effective_drivers_ratio" in s8:
    print(f"\nratio ({s8['comparison']}): {s8['effective_drivers_ratio']:.4f}")

print("\nmodel-level context:")
struct = model_interaction_structure(artifact)
print(f"  main effects {struct['n_main_effects']}, "
      f"interaction grids {struct['n_interactions']}")
if struct["all_pairwise"]:
    print(f"  {struct['note']}")

Male       top driver 42.2%   effective drivers 5.06
Female     top driver 43.9%   effective drivers 4.61

ratio (Male / Female): 1.0987

model-level context:
  main effects 0, interaction grids 55
  no single-feature components: every part of this model is a pairwise interaction, so explanation complexity varies by row rather than by which kind of component fired


That model-level line explains why this section is not an *interaction share*.

A depth-2 ensemble on real data typically produces **zero** main effects and
dozens of interaction grids, because no tree happens to split on a single
feature. An interaction share would then read 100% for everyone — a fact
about the model, not about any group. Concentration varies by row, so it
survives contact with real artifacts.

It also is not a *curvature* measure, which would suit a smooth function with
real manifold structure where a Hessian means something. A depth-2 ensemble
is piecewise constant: second derivatives vanish and finite differences
measure noise.

### §11 Counterfactual

Flip the protected attribute and see whether decisions move.

In [11]:
s11 = audit.section(11)
if s11["applicable"]:
    print(f"feature flipped      : {s11['feature']}")
    print(f"band flip rate       : {s11['band_flip_rate']:.2%}")
    print(f"mean |score change|  : {s11['mean_abs_score_change_micro']:,.0f} micro")
    for name, g in s11["groups"].items():
        print(f"  {name:<10} mean change {g['mean_score_change_micro']:+,.0f} micro")
else:
    print("not applicable —", s11["reason"])

feature flipped      : SEX
band flip rate       : 13.00%
mean |score change|  : 4,149 micro
  Male       mean change -3,739 micro
  Female     mean change +4,093 micro


Both outcomes are informative. When the attribute **is** a model input, a
nonzero band flip rate is a direct finding: applicants change bands purely
because the attribute changed.

When it is **not** an input, the section reports itself inapplicable as a
positive statement rather than skipping silently. "The model does not use it,
so the test does not apply" is the desired result and belongs in the report.

### What this example actually found

On the UCI panel the protected attribute **is** a model input, and the audit
says so from three directions that agree:

- §6 attributes **13.4% of the group score gap** directly to `SEX`
- §10 shows `SEX` as the largest single difference in *cited* adverse-action
  reason between groups
- §11 flips it and moves **13% of applicants across a band boundary**

That is a finding, not a demonstration artefact. A model carrying the
protected attribute as a feature is a straightforward fair-lending problem,
and the point of the example is that each layer detects it independently
rather than one metric being tuned to notice.

The obvious next step — dropping `SEX` and recompiling — is left to you,
because watching §11 flip to *inapplicable* and §6 redistribute that 13.4%
across the remaining features is the most instructive thing in this notebook.

---
## What was skipped, and why

A section without its inputs is recorded with the reason. A missing section
is never silent.

In [12]:
print(audit.skipped or "nothing skipped — every section had what it needed")

nothing skipped — every section had what it needed


---
## Closing

The audit reports numbers a validator can inspect and argue with. It does not
decide whether a model is fair, and it does not certify compliance with any
regulation. That boundary is deliberate, and it is printed at the foot of
every summary so it cannot be lost when the output is pasted into a report.

See [the fairness how-to](../docs/howto/fairness.md) for what each section
measures and why.